Task:
Simulate 1000 random current in the range (0.01, 1.5) in scenario A, 
given 
- current $I_1$ at position 1 of `qrodrupole focusing Lattice`(beamline 0.35878m - 0.44768m) and $I_3$ at position 3 of `qrodrupole defocusing Lattice`(beamline 0.64453m - 0.73342m),
- objective that $\alpha_x=0$ at position 8, and $\alpha_y=0$ at position 9
- weight for each objective(both 1 for `$\alpha_x=0$ at position 8` and `$\alpha_y=0$ at position 9`)

return 
- weighted MSE function $\phi(k)$
- Jacobian matrix $J(k) \in \mathbb{R}^{2 \times 2}$
- Hessian matrix $H(k) \in \mathbb{R}^{2 \times 2}$
- Eigenvalues/Eigenvectors of Hessian matrix $H(k)$

In [1]:
import numpy as np
import torch
import copy
import time
import pickle
import os, sys
current_dir=os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(current_dir, "../")))
sys.path.insert(0, os.path.abspath(os.path.join(current_dir, "../../backend")))
# print(sys.path)

# Initialize the beamline
from configs import *
from experiments_utils import load_results, save_results
from ebeam import beam as ebeam_class
from beamline import lattice
from excelElements import ExcelElements
from beamOptimizer import beamOptimizer

c:\Users\yilu2\my_files\research\FELsim\backend


compute the weighted MSE function $\phi(k)$ for scenario A, evalutated at position 8 for $\alpha_x$ and position 9 for $\alpha_y$

In [2]:
def compute_weighted_MSE(I_1, I_3, weight=[1, 1], target_alpha_x=0, target_alpha_y=0):
    """
    I_1: current at position 1 for qrodrupole focusing Lattice 1
    I_3: current at position 3 for qrodrupole defocusing Lattice 1
    weight: weight for the objectives
    target_alpha_x: target value of alpha_x at position 8
    target_alpha_y: target value of alpha_y at position 9
    """
    bl = ExcelElements(EXCEL_PATH).create_beamline()[:beamline_slice_len]
    p = PARTICLES.clone()
    obj_copy = copy.deepcopy(A_OBJ)
    opti = beamOptimizer(bl, p, noise=False)
    k_vals = [I_1, I_3]
    SP = {
        "I": {"bounds": CURRENT_BOUNDS, "start": 0.5},
        "I2": {"bounds": CURRENT_BOUNDS, "start": 0.5},
    }
    opti._prepare(A_VARS, SP, obj_copy)
    input_dict = {"I": I_1, "I2": I_3}
    k_vals = [input_dict[var_name] for var_name in opti.variablesToOptimize]
    phi=opti._optiSpeed(k_vals)
    num_goals = 2
    r_list = []
    
    stat_x = float(opti.objectives[8][0]["measured"])
    r_x = np.sqrt(weight[0] / num_goals) * (stat_x - target_alpha_x) / weight[0]
    r_list.append(r_x)
    
    stat_y = float(opti.objectives[9][0]["measured"])
    r_y = np.sqrt(weight[1] / num_goals) * (stat_y - target_alpha_y) / weight[1]
    r_list.append(r_y)
    
    r = np.array(r_list)
    return r, phi

r, phi = compute_weighted_MSE(0.7469, 0.5819, weight=[1, 1], target_alpha_x=0, target_alpha_y=0)
# print(f"R vector: {r}")
# print(f"Phi value: {phi}")

compute Jacobian matrix $J(k)$ for scenario A, with dimension $2 \times 2$, evaluated at position 8 for $\alpha_x$ and position 9 for $\alpha_y$

In [3]:
def compute_jacobian(I_1, I_3, r, delta=1e-5):
    """
    Compute the Jacobian matrix J(k) for scenario A, with dimension 2x2 
    I_1: current at position 1 for qrodrupole focusing Lattice 1
    I_3: current at position 3 for qrodrupole defocusing Lattice 1
    r: the R vector computed from compute_weighted_MSE
    delta: small perturbation for numerical differentiation
    """
    J = np.zeros((2, 2))
    
    # Perturb I_1
    r_plus_delta, _ = compute_weighted_MSE(I_1 + delta, I_3)
    r_minus_delta, _ = compute_weighted_MSE(I_1 - delta, I_3)
    
    J[:, 0] = (r_plus_delta - r_minus_delta) / (2 * delta)
    
    # Perturb I_3
    r_plus_delta, _ = compute_weighted_MSE(I_1, I_3 + delta)
    r_minus_delta, _ = compute_weighted_MSE(I_1, I_3 - delta)
    
    J[:, 1] = (r_plus_delta - r_minus_delta) / (2 * delta)
    
    return J
# print(f"Jacobian matrix J(k):\n{compute_jacobian(0.7469, 0.5819, r)}")

compute Hessian matrix based on $\phi$

In [4]:
def compute_grad_phi(I_1, I_3, r, delta=1e-5):
    return 2*compute_jacobian(I_1, I_3, r, delta).T @ r

def compute_hessian_phi(I_1, I_3, r, delta=1e-5):
    grad_phi=compute_grad_phi(I_1, I_3, r, delta)
    H = np.zeros((2, 2))
    for i in range(2):
        # Perturb I_1 and I_3
        if i == 0:
            grad_plus_delta = compute_grad_phi(I_1 + delta, I_3, r, delta)
            grad_minus_delta = compute_grad_phi(I_1 - delta, I_3, r, delta)
        else:
            grad_plus_delta = compute_grad_phi(I_1, I_3 + delta, r, delta)
            grad_minus_delta = compute_grad_phi(I_1, I_3 - delta, r, delta)
        
        H[:, i] = (grad_plus_delta - grad_minus_delta) / (2 * delta)
    return H
# test
# print(f"Gradient of phi: {compute_grad_phi(0.7469, 0.5819, r)}")
# print(f"Hessian matrix H(k):\n{compute_hessian_phi(0.7469, 0.5819, r)}")

In [5]:
def compute_eig(H):
    """
    Compute the eigenvalues and eigenvectors of the Hessian matrix H
    H: Hessian matrix
    """
    eigvals, eigvecs = np.linalg.eig(H)
    return eigvals, eigvecs
eigvals, eigvecs = compute_eig(compute_hessian_phi(0.7469, 0.5819, r))
# print(f"Eigenvalues of Hessian matrix H(k):\n{eigvals}")
# print(f"Eigenvectors of Hessian matrix H(k):\n{eigvecs}")

give a scan of current values in range (0.01, 1.5) for both $I_1$ and $I_3$, compute 
- phi
- Jacobian
- Hessian
- Eigenvalues/Eigenvectors of Hessian matrix 
Then speed up the computation by using multiprocessing.

In [6]:
# without multiprocessing
t0=time.time()
sample_num=1000
output_dict=list()
for i in range(sample_num):
    I_1 = np.random.uniform(0.01, 1.5)
    I_3 = np.random.uniform(0.01, 1.5)
    r, phi = compute_weighted_MSE(I_1, I_3, weight=[1, 1], target_alpha_x=0, target_alpha_y=0)
    J = compute_jacobian(I_1, I_3, r)
    H = compute_hessian_phi(I_1, I_3, r)
    eigvals, eigvecs = compute_eig(H)
    output_dict.append({
        "current": (I_1, I_3),
        "parameters": (phi, J, H, eigvals, eigvecs)
    })
    
t1=time.time()
print(f"Time taken without multiprocessing: {t1-t0:.4f} seconds")
# with open("../../results/scan_parameters.pkl", "wb") as f:
#     pickle.dump({
#         "output_dict": output_dict,
#     }, f)
#     print("Results saved to ../../results/scan_parameters.pkl")

Time taken without multiprocessing: 5853.9137 seconds
